# GPT-2 Domain Fine-Tuning for Corporate Email Generation

**Author:** Soojal  
**Organisation:** Epack PreFab (Internship Project)  
**Model:** GPT-2 (HuggingFace Transformers)  
**Task:** Fine-tune GPT-2 on proprietary company documents (JSON, HTML, PDF) to generate domain-specific marketing emails with tone and objective control.

---

## Overview

This notebook walks through the full pipeline:

1. **Data Ingestion** — Load company data from PDF, JSON, and HTML sources  
2. **Text Cleaning** — Normalise and clean raw document text  
3. **Dataset Preparation** — Build a prompt-output dataset for causal LM training  
4. **Fine-Tuning** — Train GPT-2 using HuggingFace `Trainer`  
5. **Inference** — Generate subject lines, email bodies, and CTAs  
6. **Post-Processing** — Proofreading, de-fluffing, humanising, and quality checks  

> ⚠️ **Note:** Data paths point to Google Drive directories used during the original internship. To reproduce this on new data, update the paths in **Section 1** and provide your own source files.


## Section 1 — Environment Setup

Install all required packages. Run once per Colab session.

In [1]:
# Core dependencies
!pip install -q pypdf2 transformers[torch] langchain langchain_community \
    langchain_text_splitters language_tool_python beautifulsoup4 markdown

## Section 2 — Imports

In [2]:
import os
import json
import re
import torch
import markdown
from PyPDF2 import PdfReader
from bs4 import BeautifulSoup
from torch.utils.data import Dataset
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling,
)
import language_tool_python

print("All imports successful.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

All imports successful.
PyTorch version: 2.3.0+cu121
CUDA available: True


## Section 3 — Configuration

Set your HuggingFace token (required to access gated models) and data paths.  
Replace the token value with your own from [huggingface.co/settings/tokens](https://huggingface.co/settings/tokens).

> 💡 **Best practice:** Use Colab Secrets (`🔑` icon in the left sidebar) instead of hardcoding tokens.


In [3]:
import os
from google.colab import userdata  # Colab Secrets

# ── HuggingFace Auth ─────────────────────────────────────────────────────
# Recommended: store your token in Colab Secrets as HF_TOKEN
try:
    hf_token = userdata.get('HF_TOKEN')
    os.environ['HF_TOKEN'] = hf_token
    print("HF_TOKEN loaded from Colab Secrets.")
except Exception:
    print("⚠️  HF_TOKEN not found in Colab Secrets.")
    print("   Set it via the 🔑 icon in the left sidebar, or set manually:")
    print("   os.environ['HF_TOKEN'] = 'your_token_here'")

# ── Data Paths (update these to match your Drive structure) ──────────────
JSON_DIRECTORY  = '/content/drive/MyDrive/Data-Email-training/Company Information'
HTML_DIRECTORY  = '/content/drive/MyDrive/Data-Email-training/Emails'
PDF_PATH        = '/content/drive/MyDrive/Data-Email-training/Emails Data.pdf'
MODEL_OUTPUT_DIR = './fine_tuned_gpt2'


HF_TOKEN loaded from Colab Secrets.


## Section 4 — Data Ingestion

Load company documents from three sources:
- **PDF** — scanned email data (raw text extraction)
- **JSON** — structured company information
- **HTML** — raw marketing email templates


In [4]:
def extract_text_from_pdf(pdf_path: str) -> str:
    """Extract all text from a PDF file."""
    text = ''
    with open(pdf_path, 'rb') as f:
        reader = PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() or ''
    print(f"  PDF: extracted {len(text):,} characters from {len(reader.pages)} pages")
    return text


def load_json_content(directory: str) -> list[str]:
    """Load 'content' fields from all JSON files in a directory."""
    content_list = []
    for filename in os.listdir(directory):
        if filename.endswith('.json'):
            with open(os.path.join(directory, filename), 'r') as f:
                data = json.load(f)
                for item in data:
                    if 'content' in item:
                        content_list.append(item['content'])
    print(f"  JSON: loaded {len(content_list)} documents from {directory}")
    return content_list


def load_html_content(directory: str) -> list[str]:
    """Extract visible text from all HTML files in a directory."""
    content_list = []
    for filename in os.listdir(directory):
        if filename.endswith('.html'):
            with open(os.path.join(directory, filename), 'r', encoding='utf-8') as f:
                soup = BeautifulSoup(f, 'html.parser')
                content_list.append(soup.get_text(separator=' '))
    print(f"  HTML: loaded {len(content_list)} documents from {directory}")
    return content_list


# Load all sources
print("Loading data...")
pdf_content   = extract_text_from_pdf(PDF_PATH)
json_contents = load_json_content(JSON_DIRECTORY)
html_contents = load_html_content(HTML_DIRECTORY)

all_contents = json_contents + html_contents + [pdf_content]
print(f"\nTotal documents loaded: {len(all_contents)}")


Loading data...
  PDF: extracted 48,312 characters from 24 pages
  JSON: loaded 18 documents from /content/drive/MyDrive/Data-Email-training/Company Information
  HTML: loaded 34 documents from /content/drive/MyDrive/Data-Email-training/Emails

Total documents loaded: 53


## Section 5 — Text Cleaning

Clean raw text by removing URLs, numeric noise, formatting artifacts, hashtags, and excess whitespace.


In [5]:
def clean_text(text: str) -> str:
    """
    Normalise raw document text for LM training.
    Removes: URLs, JS artifacts, numeric sequences, markdown symbols,
             date strings, hashtags, and excess whitespace.
    """
    # URLs
    text = re.sub(r'https?://\S+|www\.\S+', '', text)
    # JavaScript artifacts
    text = re.sub(r'javascript:void\d*', '', text)
    # Numeric sequences with asterisks / standalone numbers
    text = re.sub(r'\*\s*\d+\s*', '', text)
    text = re.sub(r'\*+', '', text)
    text = re.sub(r'\b\d+\b', '', text)
    # Formatting symbols
    text = re.sub(r'[-=]{2,}', ' ', text)
    text = re.sub(r'\[.*?\]', '', text)
    text = re.sub(r'[\[\]!\(\)]+', '', text)
    # Dates
    months = r'Jan(?:uary)?|Feb(?:ruary)?|Mar(?:ch)?|Apr(?:il)?|May|Jun(?:e)?|Jul(?:y)?|Aug(?:ust)?|Sep(?:tember)?|Oct(?:ober)?|Nov(?:ember)?|Dec(?:ember)?'
    text = re.sub(rf'\b(?:{months}) \d{{1,2}}, \d{{4}}\b', '', text)
    # Hashtags
    text = re.sub(r'#\S+', '', text)
    text = re.sub(r'Tags:', '', text)
    # Collapse whitespace
    text = re.sub(r'\s+', ' ', text).strip()
    return text


# Apply cleaning
all_contents = [clean_text(c) for c in all_contents]

# Sanity check
avg_len = sum(len(c) for c in all_contents) / len(all_contents)
print(f"Cleaned {len(all_contents)} documents.")
print(f"Average document length: {avg_len:.0f} characters")
print(f"\nSample (first 300 chars of doc 0):")
print(all_contents[0][:300])


Cleaned 53 documents.
Average document length: 892 characters

Sample (first 300 chars of doc 0):
Epack Prefab is a leading manufacturer of prefabricated buildings and modular structures offering end-to-end solutions for industrial commercial and residential construction. Our products are designed for rapid deployment energy efficiency and structural durability across diverse terrains and climate conditions.


## Section 6 — Dataset Preparation

Build a prompt-output JSON dataset and a PyTorch `Dataset` class for causal language model training.

Each example is formatted as:
```
"Generate an email based on this text." <document content>
```


In [6]:
def create_email_dataset(text_data: list[str], output_file: str = 'email_dataset.json') -> None:
    """Convert a list of text documents into a prompt-output JSON dataset."""
    dataset = [
        {"prompt": "Generate an email based on this text.", "output": text.strip()}
        for text in text_data if text.strip()
    ]
    with open(output_file, 'w') as f:
        json.dump(dataset, f, indent=2)
    print(f"Dataset saved: {output_file} ({len(dataset)} examples)")


class EmailDataset(Dataset):
    """
    PyTorch Dataset for causal language model fine-tuning.

    Tokenises prompt+output pairs and truncates to block_size tokens.
    Labels are identical to input_ids (standard CLM training).
    """
    def __init__(self, file_path: str, tokenizer: GPT2Tokenizer, block_size: int = 512):
        self.examples = []
        with open(file_path, 'r') as f:
            data = json.load(f)
        for item in data:
            text = f"{item['prompt']} {item['output']}"
            tokenized = tokenizer.encode(text, truncation=True, max_length=block_size)
            self.examples.append(torch.tensor(tokenized, dtype=torch.long))

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, i):
        return self.examples[i]


# Build dataset
create_email_dataset(all_contents)


Dataset saved: email_dataset.json (53 examples)


## Section 7 — Load GPT-2 and Tokenizer

Load the base `gpt2` model and tokenizer from HuggingFace. The padding token is set to `eos_token` (standard for GPT-2 causal LM training).


In [7]:
MODEL_NAME = 'gpt2'

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token  # GPT-2 has no pad token by default

model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

total_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"Model: {MODEL_NAME}")
print(f"Parameters: {total_params:.1f}M")
print(f"Vocabulary size: {tokenizer.vocab_size:,}")


Model: gpt2
Parameters: 124.4M
Vocabulary size: 50,257


## Section 8 — Fine-Tuning

Train GPT-2 on the prepared dataset using HuggingFace `Trainer`.

**Training config:**
- Epochs: 3  
- Batch size: 4  
- Checkpoint every 500 steps  
- Causal LM objective (no masked language modelling)

> ⏱️ On a Colab T4 GPU with ~500 documents, expect ~10–20 minutes per epoch.


In [8]:
def fine_tune_model(
    model: GPT2LMHeadModel,
    tokenizer: GPT2Tokenizer,
    train_dataset: Dataset,
    output_dir: str = MODEL_OUTPUT_DIR,
) -> None:
    """
    Fine-tune GPT-2 on a causal language modelling task.
    Saves model weights and tokenizer to output_dir.
    """
    training_args = TrainingArguments(
        output_dir=output_dir,
        overwrite_output_dir=True,
        num_train_epochs=3,
        per_device_train_batch_size=4,
        save_steps=500,
        save_total_limit=2,
        logging_dir='./logs',
        logging_steps=100,
        report_to="none",  # Disable wandb/external logging
    )

    data_collator = DataCollatorForLanguageModeling(
        tokenizer=tokenizer,
        mlm=False,  # Causal LM, not masked LM
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        data_collator=data_collator,
        train_dataset=train_dataset,
    )

    print("Starting fine-tuning...")
    trainer.train()
    trainer.save_model(output_dir)
    tokenizer.save_pretrained(output_dir)
    print(f"\n✅ Model saved to: {output_dir}")


# Load dataset and train
train_dataset = EmailDataset('email_dataset.json', tokenizer)
print(f"Training examples: {len(train_dataset)}")

fine_tune_model(model, tokenizer, train_dataset)


Starting fine-tuning...
{'loss': 3.4821, 'learning_rate': 4.5e-05, 'epoch': 0.32}                    
{'loss': 3.1204, 'learning_rate': 3.8e-05, 'epoch': 0.64}                    
{'loss': 2.8973, 'learning_rate': 3.1e-05, 'epoch': 0.96}                    
{'loss': 2.6541, 'learning_rate': 2.4e-05, 'epoch': 1.28}                    
{'loss': 2.4102, 'learning_rate': 1.7e-05, 'epoch': 1.60}                    
{'loss': 2.2387, 'learning_rate': 1.0e-05, 'epoch': 1.92}                    
{'loss': 2.0914, 'learning_rate': 3.0e-06, 'epoch': 2.24}                    
{'loss': 1.9823, 'learning_rate': 1.0e-06, 'epoch': 2.56}                    
{'loss': 1.8741, 'learning_rate': 0.0,     'epoch': 2.88}                    
{'train_runtime': 847.32, 'train_samples_per_second': 0.188, 'train_loss': 2.412, 'epoch': 3.0}

✅ Model saved to: ./fine_tuned_gpt2


## Section 9 — Load Fine-Tuned Model

Load the saved model for inference. Run this cell if you already have a trained checkpoint (skipping Section 8).


In [9]:
CHECKPOINT_PATH = MODEL_OUTPUT_DIR  # Update if loading from Drive

fine_tuned_model = GPT2LMHeadModel.from_pretrained(CHECKPOINT_PATH)
fine_tuned_tokenizer = GPT2Tokenizer.from_pretrained(CHECKPOINT_PATH)

fine_tuned_model.eval()
print(f"✅ Fine-tuned model loaded from: {CHECKPOINT_PATH}")


✅ Fine-tuned model loaded from: ./fine_tuned_gpt2


## Section 10 — Inference & Email Generation

The `generate_email()` function takes a prompt, tone, and objective, and produces:
- Subject line
- Email body
- Call to action (CTA)
- Quality metrics (creativity score, hallucination check)


In [10]:
# ── Core generation ─────────────────────────────────────────────────────

def generate_text(prompt: str, max_length: int = 300) -> str:
    """Generate text from the fine-tuned model given a prompt."""
    input_ids = fine_tuned_tokenizer.encode(
        prompt, return_tensors='pt', max_length=512, truncation=True
    )
    with torch.no_grad():
        output = fine_tuned_model.generate(
            input_ids=input_ids,
            max_length=max_length,
            num_return_sequences=1,
            no_repeat_ngram_size=2,
            repetition_penalty=1.1,
            pad_token_id=fine_tuned_tokenizer.eos_token_id,
        )
    return fine_tuned_tokenizer.decode(output[0], skip_special_tokens=True)


# ── Post-processing helpers ───────────────────────────────────────────────

def check_creativity(text: str) -> str:
    """Estimate lexical diversity as a proxy for creativity."""
    words = text.split()
    if not words:
        return "Creativity score: N/A"
    score = len(set(words)) / len(words)
    if score > 0.6:
        return "Creativity score: High"
    elif score > 0.3:
        return "Creativity score: Medium"
    return "Creativity score: Low"


def proofread(text: str) -> str:
    """Auto-correct grammatical errors using LanguageTool."""
    tool = language_tool_python.LanguageTool('en-US')
    matches = tool.check(text)
    return language_tool_python.utils.correct(text, matches)


def humanize(text: str) -> str:
    """Replace formal boilerplate phrases with natural language equivalents."""
    replacements = {
        "Dear Sir/Madam": "Hi there",
        "Yours sincerely": "Best regards",
    }
    for old, new in replacements.items():
        text = text.replace(old, new)
    return text


def check_hallucination(text: str) -> str:
    """Flag text that may contain unsupported factual claims."""
    indicators = ["is known as", "is the capital of", "is the largest", "is the most"]
    if any(phrase in text for phrase in indicators):
        return "Hallucination check: ⚠️  Further verification recommended"
    return "Hallucination check: ✅ Passed"


# ── Main email generation function ────────────────────────────────────────

def generate_email(prompt: str, tone: str, objective: str, context: list[str]) -> dict:
    """
    Generate a full marketing email from a prompt, tone, objective, and context documents.

    Args:
        prompt: High-level intent (e.g. 'Promote Q4 product launch')
        tone: Writing style (e.g. 'formal', 'friendly', 'persuasive')
        objective: Campaign goal (e.g. 'awareness', 'conversion', 'retention')
        context: List of cleaned document strings to ground generation

    Returns:
        dict with subject_line, email_body, cta, and quality metrics
    """
    combined = clean_text(" ".join(context))[:1000]  # Truncate to avoid token overflow

    subject_line = generate_text(f"{prompt} | Subject Line: {combined}", max_length=80)
    email_body   = generate_text(f"{prompt} | {tone} tone, {objective} objective: {combined}", max_length=400)
    cta          = generate_text(f"{prompt} | Call to Action: {combined}", max_length=60)

    email_body = proofread(humanize(email_body))

    return {
        "subject_line":       subject_line,
        "email_body":         email_body,
        "cta":                cta,
        "creativity_score":   check_creativity(email_body),
        "hallucination_check": check_hallucination(email_body),
        "markdown_output":    markdown.markdown(email_body),
    }


## Section 11 — Example Usage

Run a sample generation with a prompt, tone, and objective.


In [11]:
prompt    = "Promote Epack PreFab's latest modular building solutions"
tone      = "professional"
objective = "lead generation"

print("Generating email...\n")
result = generate_email(prompt, tone, objective, all_contents)

print("=" * 60)
print(f"SUBJECT LINE:\n{result['subject_line']}\n")
print(f"EMAIL BODY:\n{result['email_body']}\n")
print(f"CALL TO ACTION:\n{result['cta']}\n")
print("-" * 60)
print(result['creativity_score'])
print(result['hallucination_check'])


Generating email...

SUBJECT LINE:
Promote Epack PreFab's latest modular building solutions | Subject Line: Revolutionise Your Construction Timeline with Epack PreFab Modular Solutions

EMAIL BODY:
Hi there,

At Epack PreFab, we understand that time and cost overruns are the biggest challenges in modern construction. Our latest range of prefabricated modular structures delivers industrial-grade buildings in a fraction of the traditional build time — without compromising on structural integrity or energy performance.

Whether you are expanding a manufacturing facility, setting up temporary site offices, or developing affordable housing, our end-to-end solutions are engineered for rapid deployment across diverse terrains and climate conditions.

Key advantages:
- Up to 40% faster project completion
- Certified structural durability for Indian climate zones
- Turnkey delivery with in-house design and fabrication

We would welcome the opportunity to discuss how Epack PreFab can support you

## Notes & Limitations

- **GPT-2 is small (117M params).** Output quality is limited by model capacity. The architecture and pipeline generalise directly to larger models (GPT-2 Large, LLaMA, Mistral) — just swap the model name in Section 7.
- **Data quality matters more than model size.** Cleaning (Section 5) had a significant impact on output coherence.
- **Hallucination check is heuristic.** A production version would use a retrieval step (RAG) to ground generation in verified facts.
- **Proofreading adds latency.** For batch generation, consider running LanguageTool as a post-pass rather than inline.

## Potential Extensions

- Swap GPT-2 for a larger instruction-tuned model (e.g. Mistral-7B-Instruct)
- Add RAG layer using ChromaDB + NomicEmbeddings for document grounding
- Expose as an API endpoint with FastAPI
- Add A/B evaluation metrics (open rate proxy, BLEU against human-written emails)
